# 🕵️ Dia 45 — Análise estruturada com hipóteses concorrentes (ACH)

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 45* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-45.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Na reunião das cinco cooperativas, a analista da CredMinas concluiu: *"o mesmo `// pescador` nas cinco: é uma pessoa só"*. Rafa escreveu quatro hipóteses no quadro e disse: *"antes de concluir, a gente procura o que **derruba** cada uma"*.

Neste notebook você vai:

1. **Montar a matriz ACH do NotPetya** (o caso do original) e vê-la como mapa de calor.
2. **Pontuar inconsistência** exatamente como a fórmula `COUNTIF` do Calc.
3. **Rodar a sensibilidade** que quase todo mundo pula.
4. **Repetir tudo com o IscaKit** e escrever a checagem de premissas-chave.
5. **Jogar o quiz "isto é diagnóstico?"** e exportar a matriz para o Calc.

🐍 Só biblioteca padrão.

## 1. A matriz do NotPetya

Pergunta: *qual foi o propósito principal do surto de 27/06/2017?* Corte: fontes até **7/07/2017**.

As linhas E3, E4 e E5 seguem as avaliações resolvidas do original. As demais são **ilustrativas** (o original diz: *discuta com elas*). O formulário abaixo deixa você trocar qualquer avaliação; é para isso que ele existe.

Legenda: `CC` muito consistente · `C` consistente · `N` não se aplica · `I` inconsistente · `II` muito inconsistente.

In [ ]:
#@title Avaliações da matriz NotPetya (formato: H1,H2,H3,H4) { display-mode: "form" }
E1 = "C,C,CC,C"   #@param {type:"string"}
E2 = "C,C,CC,C"   #@param {type:"string"}
E3 = "II,C,C,C"   #@param {type:"string"}
E4 = "II,C,CC,C"  #@param {type:"string"}
E5 = "C,C,C,C"    #@param {type:"string"}
E6 = "N,N,C,C"    #@param {type:"string"}
E7 = "I,I,CC,C"   #@param {type:"string"}

HIPOTESES_NP = {
    "H1": "Ransomware financeiro, funcionando como projetado",
    "H2": "Ransomware financeiro, quebrado por engenharia ruim",
    "H3": "Destrutivo, ator ligado a Estado, disfarçado",
    "H4": "Destrutivo, ator não estatal",
}
EVIDENCIAS_NP = {  # id: (texto, fonte, data) — todas antes do corte de 7/jul/2017
    "E1": ("Entrega inicial pela atualização do M.E.Doc", "ESET; Talos", "2017-06-30"),
    "E2": ("Maioria das detecções na Ucrânia", "ESET", "2017-06-30"),
    "E3": ("E-mail de contato único, bloqueado e não substituído", "Posteo", "2017-06-27"),
    "E4": ("'Chave de instalação' aleatória: sem decifração possível", "Kaspersky", "2017-06-28"),
    "E5": ("Worm com EternalBlue/EternalRomance + credenciais", "ESET e outros", "2017-06-30"),
    "E6": ("Véspera do Dia da Constituição da Ucrânia", "Calendário", "2017-06-27"),
    "E7": ("ESET liga a operação ao TeleBots", "ESET", "2017-06-30"),
}
VALIDOS = {"CC", "C", "N", "I", "II"}

def montar(avaliacoes: dict, hipoteses: dict) -> dict:
    """Transforma 'II,C,C,C' em {'H1': 'II', 'H2': 'C', ...}, conferindo os valores."""
    matriz = {}
    for eid, texto in avaliacoes.items():
        notas = [x.strip().upper() for x in texto.split(",")]
        if len(notas) != len(hipoteses) or not set(notas) <= VALIDOS:
            raise ValueError(f"{eid}: use {len(hipoteses)} valores entre {sorted(VALIDOS)}, veio {texto!r}")
        matriz[eid] = dict(zip(hipoteses, notas))
    return matriz

MATRIZ_NP = montar({"E1": E1, "E2": E2, "E3": E3, "E4": E4, "E5": E5, "E6": E6, "E7": E7}, HIPOTESES_NP)
print("Matriz montada:", len(MATRIZ_NP), "evidências ×", len(HIPOTESES_NP), "hipóteses")

## 2. Mapa de calor, linhas cinzas e placar

O passo 4 do original: uma linha **consistente com todas as hipóteses** não tem valor diagnóstico. Aqui usamos a definição operacional: se a linha contribui com a **mesma** pontuação de inconsistência para todas as hipóteses, ela não muda a classificação, então fica cinza.

O placar é a fórmula do original: `=COUNTIF(...;"I") + 2*COUNTIF(...;"II")`.

In [ ]:
COR = {"II": "🟥", "I": "🟧", "N": "⬜", "C": "🟩", "CC": "💚"}
PESO = {"CC": 0, "C": 0, "N": 0, "I": 1, "II": 2}    # só inconsistência conta (passo 5)

def diagnostica(linha: dict) -> bool:
    """Uma linha só é diagnóstica se pesar diferente em pelo menos duas hipóteses."""
    return len({PESO[v] for v in linha.values()}) > 1

def placar(matriz: dict, ignorar=()) -> dict:
    hips = next(iter(matriz.values())).keys()
    return {h: sum(PESO[l[h]] for e, l in matriz.items() if e not in ignorar and diagnostica(l))
            for h in hips}

def desenhar(matriz, hipoteses, evidencias):
    print(" " * 6 + "  ".join(f"{h:^4}" for h in hipoteses))
    for eid, linha in matriz.items():
        celulas = "  ".join(f"{COR[linha[h]]}{linha[h]:<2}" for h in hipoteses)
        cinza = "" if diagnostica(linha) else "   ⬛ NÃO DIAGNÓSTICA (cinza)"
        print(f"{eid:<5} {celulas}   {evidencias[eid][0][:48]}{cinza}")
    p = placar(matriz)
    print("\nInconsistência  " + "  ".join(f"{h}={p[h]}" for h in hipoteses))
    melhor = min(p.values())
    print("Favorecida(s) (MENOS evidência contra):", [h for h in p if p[h] == melhor])
    return p

PLACAR_NP = desenhar(MATRIZ_NP, HIPOTESES_NP, EVIDENCIAS_NP)

> 💡 Com as avaliações padrão, **H3 e H4 empatam**. Não é defeito do notebook: é a lição do original. A evidência pública até 7/jul/2017 separa "ransomware" de "destrutivo", mas não separa "Estado" de "não Estado".

## 3. Sensibilidade (o passo que as pessoas pulam)

Três testes, como o passo 6 do original pede:

1. tirar **cada** evidência, uma de cada vez;
2. tirar as **duas** que mais pesam;
3. tratar uma evidência como **enganosa** (plantada): inverter o que ela diz.

In [ ]:
def favorecidas(p):
    m = min(p.values())
    return sorted(h for h in p if p[h] == m)

def sensibilidade(matriz):
    base = favorecidas(placar(matriz))
    print("Base:", base)
    print("\n1) Tirando uma evidência por vez:")
    for e in matriz:
        f = favorecidas(placar(matriz, {e}))
        print(f"   sem {e:<4} → {f}" + ("   ⚠️ MUDOU" if f != base else ""))
    # Peso total de cada evidência = quanto ela soma de inconsistência no placar.
    peso = {e: sum(PESO[v] for v in l.values()) for e, l in matriz.items() if diagnostica(l)}
    top2 = sorted(peso, key=peso.get, reverse=True)[:2]
    f = favorecidas(placar(matriz, set(top2)))
    print(f"\n2) Sem as duas que mais pesam {top2} → {f}" + ("   ⚠️ MUDOU" if f != base else ""))
    return top2

def enganosa(matriz, eid):
    """Se a evidência foi plantada, o que ela 'diz' se inverte: I↔C, II↔CC."""
    troca = {"I": "C", "C": "I", "II": "CC", "CC": "II", "N": "N"}
    m2 = {e: (dict((h, troca[v]) for h, v in l.items()) if e == eid else l) for e, l in matriz.items()}
    f = favorecidas(placar(m2))
    print(f"\n3) Se {eid} for enganosa → {f}")

TOP2_NP = sensibilidade(MATRIZ_NP)
enganosa(MATRIZ_NP, "E7")   # E7: e se a ligação ao TeleBots fosse uma pista plantada?

## 4. ➕ A matriz do IscaKit

Pergunta: *as campanhas do IscaKit contra as cinco cooperativas são de um único operador?* Corte: 9/out/2026, 12h00.

Os dados vêm de `caso-continuo/dados/dia-45/ach-iscakit.csv` (fictícios; avaliações **sugeridas para discutir**). A célula tenta o repositório e cai na cópia embutida sem rede.

**Antes de rodar:** decida como você avaliaria E2 (`// pescador` nas cinco ondas) e E10 (SMS também para quem nunca foi cadastrado em cooperativa nenhuma).

In [ ]:
import csv
import io
import urllib.request

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-45/ach-iscakit.csv")
EMBUTIDO = """id,evidencia,fonte,data,H1_um_operador,H2_varios_compradores,H3_original_e_imitadores,H4_interno_fornecedor
E1,As cinco ondas usam o mesmo hash de corpo HTML do IscaKit,Varredura pública (urlscan) registrada no grafo do Dia 40,2026-10-07,C,C,C,C
E2,A string '// pescador' aparece no JavaScript das cinco ondas,Captura das páginas em navegador isolado,2026-10-07,C,C,C,C
E3,Os disparos das cinco ondas ocorreram em dias diferentes mas sempre entre 07h00 e 09h00 BRT,Reclamações das cooperativas (horário dos SMS),2026-10-08,CC,I,C,C
E4,Ondas 1 a 3 usam o mesmo encurtador (lnk.example); ondas 4 e 5 usam encurtador diferente,Textos dos SMS encaminhados,2026-10-08,I,C,C,C
E5,Ondas 4 e 5 têm erros de português no texto do SMS que as ondas 1 a 3 não têm,Textos dos SMS encaminhados,2026-10-08,I,C,CC,N
E6,Print de anúncio de venda do 'IscaKit cooperativas' circulou num grupo (não verificado),Print encaminhado por terceiro,2026-10-08,I,CC,C,N
E7,Os certificados TLS das ondas 1 a 3 compartilham SAN; os das ondas 4 e 5 não,crt.sh,2026-10-07,I,C,C,N
E8,Ondas 4 e 5 hospedadas em 198.51.100.40; ondas 1 a 3 em 203.0.113.77,DNS passivo,2026-10-07,I,C,C,N
E9,As cinco cooperativas não usam o mesmo fornecedor de envio de SMS,Questionário respondido pelas cooperativas,2026-10-09,C,C,C,II
E10,Os SMS foram recebidos inclusive por números que nunca foram cadastrados em nenhuma das cooperativas,Reclamações de não cooperados repassadas às ouvidorias,2026-10-09,C,C,C,I
E11,O primeiro domínio do kit (serraazul-pix.example) é o mais antigo; os das ondas 4 e 5 foram registrados 2 dias depois de um alerta público da Serra Azul,RDAP e data do alerta,2026-10-09,C,C,CC,N
E12,As páginas de todas as ondas enviam as credenciais para o mesmo caminho /api/recebe,Captura das páginas em navegador isolado,2026-10-07,C,C,C,C
"""

try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        texto = r.read().decode("utf-8")
    print("Matriz baixada do repositório.")
except Exception as erro:
    texto = EMBUTIDO
    print(f"Sem rede ({type(erro).__name__}): usando a cópia embutida.")

linhas = list(csv.DictReader(io.StringIO(texto)))
COLS = [c for c in linhas[0] if c.startswith("H")]          # H1_um_operador, H2_..., ...
HIPOTESES_IK = {c.split("_")[0]: c.split("_", 1)[1].replace("_", " ") for c in COLS}
EVIDENCIAS_IK = {l["id"]: (l["evidencia"], l["fonte"], l["data"]) for l in linhas}
MATRIZ_IK = {l["id"]: {c.split("_")[0]: l[c] for c in COLS} for l in linhas}

# Regra do corte: nenhuma evidência depois de 9/out/2026.
fora = [e for e, (_, _, d) in EVIDENCIAS_IK.items() if d > "2026-10-09"]
print("Evidências após o corte:", fora or "nenhuma ✅")
print("Hipóteses:", HIPOTESES_IK)

In [ ]:
PLACAR_IK = desenhar(MATRIZ_IK, HIPOTESES_IK, EVIDENCIAS_IK)
print()
TOP2_IK = sensibilidade(MATRIZ_IK)
enganosa(MATRIZ_IK, "E6")   # o print do anúncio de venda nunca foi verificado

Leia o resultado devagar:

- E1, E2 e E12 ficaram **cinzas**. A "assinatura" `// pescador` é a mesma em todas as hipóteses: ela prova que o **kit** é o mesmo, e não quem o opera.
- A favorecida tem **menos evidência contra**; não é a que tem "mais a favor".
- Repare no que acontece sem E3 (horário dos disparos): é a pergunta que a matriz **não consegue responder** com segurança.

## 5. ➕ Checagem de premissas-chave

Preencha pelo menos cinco premissas do caso IscaKit. As três primeiras vêm sugeridas; a coluna "avaliação" é sua.

In [ ]:
#@title Premissas-chave do IscaKit { display-mode: "form" }
p1 = "O kit das cinco ondas é o mesmo kit, e não versões parecidas"  #@param {type:"string"}
p1_por_que = "Mesmo hash de corpo HTML e mesmo caminho /api/recebe"  #@param {type:"string"}
p1_falsa_se = "Kits diferentes que geram o mesmo HTML a partir de um modelo público"  #@param {type:"string"}
p1_aval = "com ressalvas"  #@param ["sólida", "com ressalvas", "sem apoio"]
p2 = "Os horários dos SMS refletem quando o operador trabalha"  #@param {type:"string"}
p2_por_que = "Disparos sempre entre 07h e 09h BRT"  #@param {type:"string"}
p2_falsa_se = "Disparos agendados por plataforma; horário escolhido para pegar cooperados acordando"  #@param {type:"string"}
p2_aval = "com ressalvas"  #@param ["sólida", "com ressalvas", "sem apoio"]
p3 = "O print do anúncio de venda do kit é autêntico"  #@param {type:"string"}
p3_por_que = "Circulou num grupo de analistas"  #@param {type:"string"}
p3_falsa_se = "Montagem, anúncio de outro kit, ou golpe contra golpistas"  #@param {type:"string"}
p3_aval = "sem apoio"  #@param ["sólida", "com ressalvas", "sem apoio"]
p4 = ""  #@param {type:"string"}
p4_por_que = ""  #@param {type:"string"}
p4_falsa_se = ""  #@param {type:"string"}
p4_aval = "com ressalvas"  #@param ["sólida", "com ressalvas", "sem apoio"]
p5 = ""  #@param {type:"string"}
p5_por_que = ""  #@param {type:"string"}
p5_falsa_se = ""  #@param {type:"string"}
p5_aval = "com ressalvas"  #@param ["sólida", "com ressalvas", "sem apoio"]

PREMISSAS = [(p1, p1_por_que, p1_falsa_se, p1_aval), (p2, p2_por_que, p2_falsa_se, p2_aval),
             (p3, p3_por_que, p3_falsa_se, p3_aval), (p4, p4_por_que, p4_falsa_se, p4_aval),
             (p5, p5_por_que, p5_falsa_se, p5_aval)]
preenchidas = [p for p in PREMISSAS if p[0].strip()]
print(f"{len(preenchidas)} premissas preenchidas" + ("" if len(preenchidas) >= 5 else " — o checkpoint pede 5"))
for prem, porque, falsa, aval in preenchidas:
    print(f"  • [{aval}] {prem}\n      por quê: {porque}\n      falsa se: {falsa}")

## 6. 🎮 Quiz: isto é diagnóstico?

Para cada evidência do caso, responda **S** se ela ajuda a separar as hipóteses H1–H4 do IscaKit, ou **N** se ela é consistente com todas. O gabarito vem da própria matriz.

In [ ]:
MODO_JOGO = True

acertos = 0
for i, (eid, linha) in enumerate(MATRIZ_IK.items(), start=1):
    certo = diagnostica(linha)
    print(f"\n[{i:02d}/{len(MATRIZ_IK)}] {eid}: {EVIDENCIAS_IK[eid][0]}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Diagnóstica? (S/N): ").strip().upper()
        acertos += ((r == "S") == certo)
        print("   ✅" if (r == "S") == certo else "   ❌")
    print("   Gabarito:", "DIAGNÓSTICA" if certo else "NÃO diagnóstica (consistente com todas)",
          "—", " ".join(f"{h}:{v}" for h, v in linha.items()))
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(MATRIZ_IK)}")

## 7. 📝 Conclusão e exportação

O parágrafo de conclusão do passo 6, para os dois casos. A célula confere se você classificou **todas** as hipóteses e se disse o que a matriz **não** responde; depois exporta as matrizes em CSV (abra no LibreOffice e aplique a formatação condicional do original).

In [ ]:
#@title Conclusões { display-mode: "form" }
conclusao_np = "H3 e H4 empatam com menor inconsistência (0), à frente de H2 (1) e H1 (5). E3 e E4 decidem contra H1. Se E7 for plantada, H3 perde a vantagem. A matriz não separa H3 de H4 com a evidência de até 7/jul/2017."  #@param {type:"string"}
conclusao_ik = "H3 (original e imitadores) tem a menor inconsistência (0), seguida de H2 (1), H4 (3) e H1 (5). Sem E3, H2 e H3 empatam: a matriz não separa compradores independentes de imitadores. A string // pescador não é diagnóstica."  #@param {type:"string"}

for nome, texto, hips in (("NotPetya", conclusao_np, HIPOTESES_NP), ("IscaKit", conclusao_ik, HIPOTESES_IK)):
    falta = [h for h in hips if h not in texto]
    nao_responde = any(p in texto.lower() for p in ("não separa", "nao separa", "não responde", "não consegue"))
    print(f"{nome}: hipóteses citadas {'✅' if not falta else '❌ faltam ' + str(falta)} | "
          f"diz o que não responde {'✅' if nao_responde else '❌'}")

import pathlib
pathlib.Path("notas").mkdir(exist_ok=True)
for nome, matriz, evid in (("notpetya", MATRIZ_NP, EVIDENCIAS_NP), ("iscakit", MATRIZ_IK, EVIDENCIAS_IK)):
    hips = list(next(iter(matriz.values())))
    with open(f"notas/dia45-ach-{nome}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "evidencia", "fonte", "data", *hips, "diagnostica"])
        for e, l in matriz.items():
            w.writerow([e, *evid[e], *(l[h] for h in hips), "sim" if diagnostica(l) else "não (cinza)"])
        p = placar(matriz)
        w.writerow(["placar", "I=1, II=2", "", "", *(p[h] for h in hips), ""])
pathlib.Path("notas/dia45-conclusoes.md").write_text(
    f"# Conclusões ACH — Dia 45\n\n## NotPetya\n{conclusao_np}\n\n## IscaKit\n{conclusao_ik}\n", encoding="utf-8")
print("💾 notas/dia45-ach-notpetya.csv, notas/dia45-ach-iscakit.csv, notas/dia45-conclusoes.md")

try:
    from google.colab import files
    for n in ("notas/dia45-ach-notpetya.csv", "notas/dia45-ach-iscakit.csv", "notas/dia45-conclusoes.md"):
        files.download(n)
except ImportError:
    pass

## ✅ Antes de fechar o Dia 45

- [ ] Matriz do NotPetya com pelo menos 10 evidências no Calc (aqui são 7: acrescente 3 das fontes do original)
- [ ] Pelo menos uma linha cinza, e você sabe explicar por quê
- [ ] Sensibilidade rodada: o que muda sem os dois itens principais?
- [ ] Matriz do IscaKit com conclusão que classifica H1–H4 e diz o que não responde
- [ ] Cinco premissas-chave, cada uma avaliada
- [ ] Só depois: leitura do comunicado do Reino Unido de 2018 e comparação

**Amanhã (Dia 46):** a Marta precisa de uma página. Você transforma a ACH num **informe de inteligência** com linguagem calibrada: probabilidade e confiança, em frases separadas.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>